In [ ]:
# <editor-fold desc="Imports">
import os
import time
import torch
import warnings
from tqdm import tqdm

import nibabel as nib
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
# </editor-fold>

In [ ]:

# <editor-fold desc="Clinical Data Preparation and Cleaning">

# Default dataframe cleaning
clinical_data_path = "/scratch/tgoedietdoebe/AI-project/data/labels_BinClass.csv"
clinical_data = pd.read_csv(clinical_data_path)
clinical_data.dropna(inplace=True)
clinical_data.reset_index(drop=True, inplace=True)
clinical_data['w8_responder'] = clinical_data['w8_responder'].map({'Yes': 1, 'No': 0})
clinical_data['Stage1TX'] = clinical_data['Stage1TX'].map({'SER': 1, 'PLA': 0})
subjects_to_remove = ['CU0058', 'CU0059', 'CU0060', 'CU0061', 'CU0068', 'CU0074']
filtered_clinical_data = clinical_data[~clinical_data['ProjectSpecificId'].isin(subjects_to_remove)]
filtered_clinical_data.reset_index(drop=True, inplace=True)
sertraline_df = filtered_clinical_data[filtered_clinical_data['Stage1TX'] == 1]


fmri_data_path = "/data/projects/depredict/repositories/EMBARC/data/data_bids/derivatives/_fmriprep/output/"
subject_ids = sertraline_df['ProjectSpecificId'].tolist()
fmri_files = {}
missing_subjects = []
for sub_id in subject_ids:
    file_pattern = os.path.join(fmri_data_path,
                                f"sub-{sub_id}/ses-1/func/sub-{sub_id}_ses-1_task-rest1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")
    if os.path.exists(file_pattern):
        fmri_files[sub_id] = file_pattern
    else:
        missing_subjects.append(sub_id)
processed_df = sertraline_df[sertraline_df['ProjectSpecificId'].isin(fmri_files.keys())]
processed_df = processed_df.copy()
processed_df['fMRI_path'] = processed_df['ProjectSpecificId'].map(fmri_files)
processed_df
# </editor-fold>

In [ ]:

# <editor-fold desc="Data Classes & Functions">
class fMRIDataset(Dataset):
    def __init__(self, df, data_dict, transform=None):
        """
        df: DataFrame with columns: ['subject_part', 'label']
        data_dict: {subject_part_id: 3D numpy array or torch.Tensor}
        """
        self.data = df.reset_index(drop=True)
        self.data_dict = data_dict
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        subject_part = row['subject_part']
        label = row['label']

        image = self.data_dict[subject_part]  # Already precomputed

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

def preload_mean_images(df, mean_image_path):
    data_dict = {}
    part_paths = []

    # Step 1: Build list of expected files (both parts)
    for subject_id in df["ProjectSpecificId"]:
        part_paths.append((f"{subject_id}_part1.pt", f"{subject_id}_part1"))
        part_paths.append((f"{subject_id}_part2.pt", f"{subject_id}_part2"))
        part_paths.append((f"{subject_id}_part3.pt", f"{subject_id}_part3"))
        part_paths.append((f"{subject_id}_part4.pt", f"{subject_id}_part4"))

    # Step 2: Load each .pt file individually
    for filename, part_id in tqdm(part_paths, desc="Preloading mean images"):
        full_path = os.path.join(mean_image_path, filename)
        if not os.path.exists(full_path):
            print(f"⚠️ Missing: {filename}")
            continue

        tensor = torch.load(full_path).float()
        data_dict[part_id] = tensor

    return data_dict

# </editor-fold>


In [ ]:

# <editor-fold desc="Precompute Mean Images">
def compute_and_normalize_mean(tensor, eps=1e-6):
    """Compute the mean image and normalize it."""
    mean_image = tensor.mean(dim=0)  # Compute mean along time dimension
    mean_image = (mean_image - mean_image.mean()) / (mean_image.std() + eps)  # Normalize
    return mean_image.unsqueeze(0)  # Add channel dimension

def save_mean_images(mean_images, subject_id, save_dir):
    """Save the mean images to disk."""
    for i, mean_image in enumerate(mean_images, start=1):
        save_path = os.path.join(save_dir, f"{subject_id}_part{i}.pt")
        torch.save(mean_image, save_path)

def precompute_mean_images(data_dict, save_dir="/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/T_45/"):
    os.makedirs(save_dir, exist_ok=True)
    for subject_id, fmri_tensor in tqdm(data_dict.items(), desc="Precomputing mean images"):
        if fmri_tensor.shape[0] < 180:
            print(f"Skipping {subject_id}: Not enough timepoints ({fmri_tensor.shape[0]}).")
            continue

        # Split into T='45' timepoints and process each segment
        segments = [fmri_tensor[i:i+45] for i in range(0, 180, 45)]
        mean_images = [compute_and_normalize_mean(segment) for segment in segments]

        # Save the mean images
        save_mean_images(mean_images, subject_id, save_dir)
        print(f"Saved mean images for {subject_id}")

# def precompute_mean_images(data_dict, save_dir="/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/T_45/"):
#     os.makedirs(save_dir, exist_ok=True)
#     for subject_id, fmri_tensor in tqdm(data_dict.items(), desc="Precomputing mean images"):
#         # Ensure the tensor has at least 180 timepoints
#         if fmri_tensor.shape[0] < 180:
#             print(f"Skipping {subject_id}: Not enough timepoints ({fmri_tensor.shape[0]}).")
#             continue
#
#         # Split into T='45'  timepoints
#         T1 = fmri_tensor[:45]  # Shape: [90, D, H, W]
#         T2 = fmri_tensor[45:90]  # Shape: [90, D, H, W]
#         T3 = fmri_tensor[90:135]  # Shape: [90, D, H, W]
#         T4 = fmri_tensor[135:180]  # Shape: [90, D, H, W]
#
#         # Compute mean images
#         mean_image_1 = T1.mean(dim=0)  # Shape: [D, H, W]
#         mean_image_2 = T2.mean(dim=0)  # Shape: [D, H, W]
#         mean_image_3 = T3.mean(dim=0)  # Shape: [D, H, W]
#         mean_image_4 = T4.mean(dim=0)  # Shape: [D, H, W]
#
#         # Normalize the mean images
#         eps = 1e-6
#         mean_image_1 = (mean_image_1 - mean_image_1.mean()) / (mean_image_1.std() + eps)
#         mean_image_2 = (mean_image_2 - mean_image_2.mean()) / (mean_image_2.std() + eps)
#         mean_image_3 = (mean_image_3 - mean_image_3.mean()) / (mean_image_3.std() + eps)
#         mean_image_4 = (mean_image_4 - mean_image_4.mean()) / (mean_image_4.std() + eps)
#
#         # Add a channel dimension to match model input shape
#         mean_image_1 = mean_image_1.unsqueeze(0)  # Shape: [1, D, H, W]
#         mean_image_2 = mean_image_2.unsqueeze(0)  # Shape: [1, D, H, W]
#         mean_image_3 = mean_image_3.unsqueeze(0)  # Shape: [1, D, H, W]
#         mean_image_4 = mean_image_4.unsqueeze(0)  # Shape: [1, D, H, W]
#
#         # Save the mean images
#         save_path_1 = os.path.join(save_dir, f"{subject_id}_part1.pt")
#         save_path_2 = os.path.join(save_dir, f"{subject_id}_part2.pt")
#         save_path_3 = os.path.join(save_dir, f"{subject_id}_part3.pt")
#         save_path_4 = os.path.join(save_dir, f"{subject_id}_part4.pt")
#
#         torch.save(mean_image_1, save_path_1)
#         torch.save(mean_image_2, save_path_2)
#         torch.save(mean_image_3, save_path_3)
#         torch.save(mean_image_4, save_path_4)
#
#         mean_saves = []
#
#         print(f"Saved mean images for {subject_id} at {mean_saves}")
# </editor-fold>

In [ ]:

# <editor-fold desc="Model Definition">
class Simple3DCNN(nn.Module):
    def __init__(self):
        super(Simple3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)

        # Use Global Average Pooling to reduce feature map size efficiently
        self.global_pool = nn.AdaptiveAvgPool3d((4, 4, 4))

        # FC Layer adjusted for the reduced size
        self.fc1 = nn.Linear(64 * 4 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        # Reduce feature map size while retaining granularity
        x = self.global_pool(x)

        x = x.view(x.size(0), -1)  # Flatten for FC layer
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class HighAccuracy3DCNN(nn.Module):
    def __init__(self):
        super(HighAccuracy3DCNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm3d(8),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv3d(8, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.feature_dim = 64 * 6 * 7 * 6

        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(self.feature_dim, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv1(x)  # [B, 8, ~48, ~57, ~48]
        x = self.conv2(x)  # [B, 16, ~24, ~28, ~24]
        x = self.conv3(x)  # [B, 32, ~12, ~14, ~12]
        x = self.conv4(x)  # [B, 64, ~6, ~7, ~6]

        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # No sigmoid here — use BCEWithLogitsLoss

        return x
# </editor-fold>


In [ ]:

# <editor-fold desc="Training & Evaluation Loop">

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    warnings.filterwarnings("ignore", category=FutureWarning)

    history = {
        'epoch': [],
        'train_loss': [],
        'train_accuracy': [],
        'train_f1': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': []
    }
    batch_check = True
    total_start_time = time.time()

    for epoch in range(num_epochs):
        print(f"\n🚀 Starting Epoch {epoch + 1}/{num_epochs}")
        model.train()
        total_loss = 0
        all_preds = []
        all_labels = []


        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Train Epoch {epoch+1}'):

            if batch_check:
                print(f"Batch shape: {images.shape}, labels: {labels.shape}")
                batch_check = False

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            with torch.no_grad():
                preds = (torch.sigmoid(outputs) > 0.5).int()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Compute training metrics
        train_loss = total_loss / len(train_loader)
        train_accuracy = (np.array(all_preds) == np.array(all_labels)).mean()
        try:
            train_f1 = f1_score(all_labels, all_preds)
        except:
            train_f1 = 0.0

        # Validation
        val_loss, val_accuracy, val_precision, val_recall, val_f1 = evaluate_model(model, val_loader, criterion)

        # Store metrics
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['val_precision'].append(val_precision)
        history['val_recall'].append(val_recall)
        history['val_f1'].append(val_f1)

        # Log
        print(f"\n📊 Epoch {epoch + 1} Summary:")
        print(f"   Train Loss: {train_loss:.4f} | Accuracy: {train_accuracy*100:.2f}% | F1: {train_f1:.4f}")
        print(f"   Val Loss: {val_loss:.4f} | Accuracy: {val_accuracy*100:.2f}%")
        print(f"   Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")

    total_time = time.time() - total_start_time
    print(f"\n⏱️ Total Training Time: {total_time:.2f} seconds")

    return history

def evaluate_model(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels.float())
            total_loss += loss.item()

            predicted = (torch.sigmoid(outputs) > 0.5).int()
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_loss / len(val_loader)
    accuracy = (np.array(all_predictions) == np.array(all_labels)).mean()
    try:
        precision = precision_score(all_labels, all_predictions)
        recall = recall_score(all_labels, all_predictions)
        f1 = f1_score(all_labels, all_predictions)
    except:
        precision = recall = f1 = 0.0

    return avg_val_loss, accuracy, precision, recall, f1

# </editor-fold>


In [ ]:

#<editor-fold desc="Non-Slurm_1 Execution">
# Path to precomputed mean .pt files
mean_image_path = "/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/T_45/"

# Load precomputed mean image tensors
# data_dict = preload_mean_images(processed_df, mean_image_path)  # This now returns {subject_part: tensor}
train_val_df, test_df = train_test_split(processed_df, test_size=0.2, random_state=42, stratify=processed_df['w8_responder'])

data_dict = preload_mean_images(train_val_df, mean_image_path)

mean_df = pd.DataFrame([
    {"subject_part": part_id,
     "label": int(processed_df.loc[processed_df["ProjectSpecificId"] == part_id.split("_part")[0], "w8_responder"].values[0])
    }
    for part_id in data_dict.keys()
])

test_data_dict = preload_mean_images(test_df, mean_image_path)

test_mean_df = pd.DataFrame([
    {
        "subject_part": part_id,
        "label": int(test_df.loc[test_df["ProjectSpecificId"] == part_id.split("_part")[0], "w8_responder"].values[0])
    }
    for part_id in test_data_dict.keys()
])

test_dataset = fMRIDataset(test_mean_df, test_data_dict)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

mean_df
# </editor-fold>

In [ ]:

#<editor-fold desc="Non-Slurm_2 Execution">

# Set up device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_histories = []
# </editor-fold>

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader

# … assume your Simple3DCNN / HighAccuracy3DCNN, train_model, evaluate_model,
#    and fMRIDataset are already defined above this cell …

def run_cv_and_save(
    run_id: int,
    mean_df,
    data_dict,
    test_loader,
    device,
    n_splits:      int   = 5,
    num_epochs:    int   = 5,
    learning_rate: float = 0.0001,
    batch_size:    int   = 8,
    base_outdir:   str   = "./cv_runs/"
, nruns=2):
    outdir = os.path.join(base_outdir, f"run_{run_id}")
    os.makedirs(outdir, exist_ok=True)

    # ── 1) per‐epoch log for this run ─────────────────────────────
    epoch_log_path = os.path.join(outdir, "training_log.csv")
    with open(epoch_log_path, "w") as f:
        f.write(
            "fold,epoch,"
            "train_loss,train_acc,train_f1,"
            "val_loss,val_acc,val_prec,val_rec,val_f1\n"
        )

    # ── 2)  K-Fold CV ────────────────────────────────────────────
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    all_histories = []

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(mean_df, mean_df["label"]), start=1):

        # — split into fold DataFrames & DataLoaders —
        train_df = mean_df.iloc[train_idx].reset_index(drop=True)
        val_df   = mean_df.iloc[val_idx].reset_index(drop=True)

        train_loader = DataLoader(
            fMRIDataset(train_df, data_dict),
            batch_size=batch_size, shuffle=True
        )
        val_loader   = DataLoader(
            fMRIDataset(val_df,   data_dict),
            batch_size=batch_size
        )

        # — model + optimizer + loss —
        model     = HighAccuracy3DCNN().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        criterion = torch.nn.BCEWithLogitsLoss()

        # — train this fold —
        history = train_model(
            model, train_loader, val_loader,
            criterion, optimizer, num_epochs
        )
        all_histories.append(history)

        # — append fold’s per-epoch metrics —
        with open(epoch_log_path, "a") as f:
            for e in range(num_epochs):
                f.write(",".join(map(str, [
                    fold,
                    e+1,
                    history["train_loss"][e],
                    history["train_accuracy"][e],
                    history["train_f1"][e],
                    history["val_loss"][e],
                    history["val_accuracy"][e],
                    history["val_precision"][e],
                    history["val_recall"][e],
                    history["val_f1"][e],
                ])) + "\n")

    # ── 3) final hold-out test evaluation ───────────────────────
    test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate_model(
        model, test_loader, criterion
    )

    # — print to console —
    print("\n🔍 Final Evaluation on True Hold-out Test Set")
    print(f"✅ Test Set Results: "
          f"Accuracy: {test_acc:.4f} | "
          f"Precision: {test_prec:.4f} | "
          f"Recall: {test_rec:.4f} | "
          f"F1: {test_f1:.4f}")

    # — append to master summary CSV —
    master_log = os.path.join(base_outdir, "master_test_results.csv")
    if not os.path.exists(master_log):
        with open(master_log, "w") as f:
            f.write("run,accuracy,precision,recall,f1\n")
    with open(master_log, "a") as f:
        f.write(f"{run_id},{test_acc:.4f},{test_prec:.4f},{test_rec:.4f},{test_f1:.4f}\n")
        f.flush()

    # ── 4) save averaged CV plots for this run ──────────────────
    epochs = all_histories[0]["epoch"]
    avg    = lambda key: np.mean([h[key] for h in all_histories], axis=0)

    # (a) Train vs Val Loss
    plt.figure(figsize=(7, 5))
    plt.plot(epochs, avg("train_loss"), label="Train Loss")
    plt.plot(epochs, avg("val_loss"),   label="Val Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title(f"Run {run_id}: Avg Train/Val Loss")
    plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(outdir, "avg_loss_curve.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

    # (b) Validation Accuracy
    plt.figure(figsize=(7, 5))
    plt.plot(epochs, avg("val_accuracy") * 100, label="Val Accuracy (%)")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)")
    plt.title(f"Run {run_id}: Avg Val Accuracy")
    plt.grid(True)
    plt.savefig(os.path.join(outdir, "avg_val_accuracy.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

    # ── (c) Precision / Recall / F1 ─────────────────────────────
    plt.figure(figsize=(7, 5))
    plt.plot(epochs, avg("val_precision"), label="Precision")
    plt.plot(epochs, avg("val_recall"),    label="Recall")
    plt.plot(epochs, avg("val_f1"),        label="F1 Score")
    plt.xlabel("Epoch"); plt.ylabel("Score")
    plt.title(f"Run {run_id}: Avg Val Precision/Recall/F1")
    plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(outdir, "avg_prf1.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # ── (d) Per-Fold Validation Accuracy ────────────────────────
    plt.figure(figsize=(7, 5))
    for i, h in enumerate(all_histories, start=1):
        plt.plot(h["epoch"], [v*100 for v in h["val_accuracy"]],
                 label=f"Fold {i}")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)")
    plt.title(f"Run {run_id}: Val Accuracy per Fold")
    plt.legend(ncol=2); plt.grid(True)
    plt.savefig(os.path.join(outdir, "per_fold_val_accuracy.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

    # ── (e) Per-Fold F1 Score ────────────────────────────────────
    plt.figure(figsize=(7, 5))
    for i, h in enumerate(all_histories, start=1):
        plt.plot(h["epoch"], h["val_f1"], label=f"Fold {i}")
    plt.xlabel("Epoch"); plt.ylabel("F1 Score")
    plt.title(f"Run {run_id}: Val F1 per Fold")
    plt.legend(ncol=2); plt.grid(True)
    plt.savefig(os.path.join(outdir, "per_fold_val_f1.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

    # ── (f) Metric Variance Across Folds (±1 std) ───────────────
    plt.figure(figsize=(7, 5))
    for key, ylabel in [
        ("val_accuracy", "Accuracy (%)"),
        ("val_precision", "Precision"),
        ("val_recall",    "Recall"),
        ("val_f1",        "F1 Score")
    ]:
        arr = np.array([h[key] for h in all_histories])
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        xs   = epochs
        ys   = mean * (100 if key=="val_accuracy" else 1)
        ss   = std  * (100 if key=="val_accuracy" else 1)

        plt.plot(xs, ys, label=f"{ylabel} Mean")
        plt.fill_between(xs, ys-ss, ys+ss, alpha=0.2,
                         label=f"{ylabel} ±1 std")
    plt.xlabel("Epoch"); plt.ylabel("Score")
    plt.title(f"Run {run_id}: Metric Variance Across Folds")
    plt.legend(ncol=1); plt.grid(True)
    plt.savefig(os.path.join(outdir, "metric_variance.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

    # ── (g) ROC Curve on Test Set ────────────────────────────────
    # first gather all test outputs & labels
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for imgs, labs in test_loader:
            imgs = imgs.to(device)
            logits = model(imgs).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            y_score.extend(probs.tolist())
            y_true.extend(labs.numpy().tolist())

    from sklearn.metrics import roc_curve, auc
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc      = auc(fpr, tpr)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"Run {run_id}: ROC Curve (Test Set)")
    plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(outdir, "roc_curve.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

    # ── (h) Confusion Matrix on Test Set ─────────────────────────
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    y_pred = [1 if p>0.5 else 0 for p in y_score]
    cm     = confusion_matrix(y_true, y_pred)
    disp   = ConfusionMatrixDisplay(cm)
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax)
    plt.title(f"Run {run_id}: Confusion Matrix (Test Set)")
    plt.savefig(os.path.join(outdir, "confusion_matrix.png"),
                dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✅ Run {run_id} completed and saved to {outdir}")


In [ ]:
n_runs = 1000
for run_id in range(1, n_runs + 1):
    print(f"▶️ Starting run {run_id}/{n_runs}")
    run_cv_and_save(
        run_id=run_id,   # ← pass the loop variable here
        mean_df=mean_df,
        data_dict=data_dict,
        test_loader=test_loader,
        device=device,
        n_splits=10,
        num_epochs=10,        # ← number of epochs
        learning_rate=1e-4,  # ← tweak learning rate
        batch_size=8,        # ← tweak batch size
        base_outdir="/scratch/tgoedietdoebe/AI-project/data/processed/mean_image/T_45/outputs/"
    )


Train Epoch 1: 100%|██████████| 38/38 [00:17<00:00,  2.16it/s]



📊 Epoch 1 Summary:
   Train Loss: 0.7182 | Accuracy: 55.30% | F1: 0.5545
   Val Loss: 0.6407 | Accuracy: 55.88%
   Precision: 1.0000 | Recall: 0.1667 | F1: 0.2857

🚀 Starting Epoch 2/10


Train Epoch 2: 100%|██████████| 38/38 [00:17<00:00,  2.14it/s]



📊 Epoch 2 Summary:
   Train Loss: 0.5699 | Accuracy: 75.17% | F1: 0.7734
   Val Loss: 0.5425 | Accuracy: 76.47%
   Precision: 1.0000 | Recall: 0.5556 | F1: 0.7143

🚀 Starting Epoch 3/10


Train Epoch 3: 100%|██████████| 38/38 [00:17<00:00,  2.15it/s]


...

Train Epoch 2: 100%|██████████| 38/38 [00:16<00:00,  2.26it/s]



📊 Epoch 2 Summary:
   Train Loss: 0.6484 | Accuracy: 59.41% | F1: 0.6215
   Val Loss: 0.5146 | Accuracy: 63.64%
   Precision: 1.0000 | Recall: 0.2941 | F1: 0.4545

🚀 Starting Epoch 3/10


Train Epoch 3: 100%|██████████| 38/38 [00:16<00:00,  2.24it/s]



📊 Epoch 3 Summary:
   Train Loss: 0.4956 | Accuracy: 78.22% | F1: 0.8012
   Val Loss: 0.4321 | Accuracy: 87.88%
   Precision: 0.8421 | Recall: 0.9412 | F1: 0.8889

🚀 Starting Epoch 4/10


Train Epoch 4:  63%|██████▎   | 24/38 [00:10<00:06,  2.28it/s]

In [ ]:
# adjust train loop so it saves or updates some file so it can be interrupted at any point